[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/ModelsOfTheMotorSystems/blob/master/In%20class%20exercises/Ex5_Multisensory_Feedback.ipynb)

# Lecture 5 In-Class Assignment: Bayesian Multisensory Integration

In the lecture, perceptual illusions showed that perception is an inference process:
the nervous system combines incoming sensory evidence with prior expectations about
the world. In this exercise we will model that idea with Gaussian Bayesian
estimation.

We will work through three related questions:
1. How should a prior expectation and one noisy sensory observation be combined?
2. How should the estimate change after several repeated observations?
3. How should two sensory modalities be weighted when one is more reliable?

**Goal**: Use reliability-weighted averaging to predict the posterior estimate and
uncertainty for simple multisensory examples.

## Outline
1. [Imports & Setup](#section1)
2. [One Observation](#section2)
3. [Repeated Observations](#section3)
4. [Two Sensory Modalities](#section4)
5. [Reliability and Bias](#section5)
6. [Discussion](#section6)

**Instructions**:
- Cells marked **DO NOT EDIT** provide helper functions and plotting code.
- Cells marked **STUDENT CODE HERE** are the places to complete the exercise.
- Keep units in Newtons (N) for the box-weight examples.

<a id="section1"></a>
## 1) Imports & Setup

**DO NOT EDIT**: Basic imports, plotting defaults, and plotting helpers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
def gaussian_pdf(x, mu, sigma):
    """Probability density for a Gaussian distribution."""
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


def plot_gaussian_update(mu_prior, sigma_prior, mu_data, sigma_data, mu_post, sigma_post):
    """Plot prior, likelihood, and posterior on the same axis."""
    x_min = min(mu_prior - 4 * sigma_prior, mu_data - 4 * sigma_data, mu_post - 4 * sigma_post)
    x_max = max(mu_prior + 4 * sigma_prior, mu_data + 4 * sigma_data, mu_post + 4 * sigma_post)
    x = np.linspace(x_min, x_max, 500)

    plt.plot(x, gaussian_pdf(x, mu_prior, sigma_prior), label="Prior")
    plt.plot(x, gaussian_pdf(x, mu_data, sigma_data), label="Likelihood")
    plt.plot(x, gaussian_pdf(x, mu_post, sigma_post), label="Posterior")
    plt.xlabel("Weight estimate (N)")
    plt.ylabel("Probability density")
    plt.legend()
    plt.tight_layout()


def print_estimate(label, mu, sigma):
    """Print an estimate in a consistent format."""
    print(f"{label}: mean = {mu:.3f} N, sigma = {sigma:.3f} N")

<a id="section2"></a>
## 2) One Observation

You see a box and expect it to be fairly light:
- Prior mean: $\mu_0 = 3.5$ N
- Prior uncertainty: $\sigma_0 = 1.0$ N

When you lift it once, the noisy sensory measurement feels heavier:
- Observation: $x = 6.0$ N
- Sensory uncertainty: $\sigma_x = 1.0$ N

For two independent Gaussian estimates, the useful quantity is **precision**:

$$
\text{precision} = \frac{1}{\sigma^2}
$$

The posterior is a precision-weighted average:

$$
\sigma_{post}^2 = \frac{1}{p_0 + p_x}
$$

$$
\mu_{post} = \sigma_{post}^2(p_0\mu_0 + p_x x)
$$

**Student Task**: Replace the simple placeholder below with the Bayesian
posterior mean and standard deviation after this one lift.

In [ ]:
# STUDENT CODE HERE
mu_prior = 3.5
sigma_prior = 1.0
mu_lift = 6.0
sigma_lift = 1.0

mu_post_1 = mu_lift
sigma_post_1 = sigma_lift

print_estimate("Posterior after one lift", mu_post_1, sigma_post_1)

In [ ]:
plot_gaussian_update(mu_prior, sigma_prior, mu_lift, sigma_lift, mu_post_1, sigma_post_1)
plt.title("Combining a prior expectation with one lift")
plt.show()

**Questions**:
- Is the posterior closer to the prior or the sensory observation?
- Why did the uncertainty get smaller after combining the two sources?

<a id="section3"></a>
## 3) Repeated Observations

You continue to lift the same box. Your next sensory measurements are:

```python
6, 7, 7, 4, 5
```

Assume each observation has the same sensory noise, $\sigma_x = 1.0$ N.

**Student Task**: Starting from the original prior, update the estimate one
observation at a time. The loop currently treats each measurement as the whole
posterior. Replace that with a Bayesian update.

In [ ]:
measurements = np.array([6.0, 7.0, 7.0, 4.0, 5.0])
sigma_measurement = 1.0

In [ ]:
current_mu = mu_prior
current_sigma = sigma_prior

posterior_means = []
posterior_sigmas = []

for measurement in measurements:
    # STUDENT CODE HERE
    current_mu = measurement
    current_sigma = sigma_measurement
    posterior_means.append(current_mu)
    posterior_sigmas.append(current_sigma)

posterior_means = np.array(posterior_means)
posterior_sigmas = np.array(posterior_sigmas)

print_estimate("Final posterior", posterior_means[-1], posterior_sigmas[-1])

In [ ]:
lift_numbers = np.arange(1, len(measurements) + 1)

plt.errorbar(lift_numbers, posterior_means, yerr=posterior_sigmas, marker="o", capsize=4)
plt.axhline(mu_prior, linestyle="--", color="0.4", label="Original prior mean")
plt.scatter(lift_numbers, measurements, color="tab:orange", label="Measurements")
plt.xlabel("Lift number")
plt.ylabel("Weight estimate (N)")
plt.title("Posterior estimate after repeated lifts")
plt.legend()
plt.tight_layout()
plt.show()

**Questions**:
- How does the estimate move as more measurements arrive?
- What happens to the posterior uncertainty?
- How is this related to motor adaptation after repeated exposure?

<a id="section4"></a>
## 4) Two Sensory Modalities

Multisensory integration uses the same logic. Suppose vision and haptics provide
two different estimates of an object's weight:
- Vision predicts $4.0$ N with uncertainty $\sigma_v = 1.5$ N.
- Haptics predicts $6.0$ N with uncertainty $\sigma_h = 0.5$ N.

**Student Task**: Combine these two estimates. The placeholder below uses only
haptics. Replace it with a reliability-weighted posterior.

In [ ]:
mu_visual = 4.0
sigma_visual = 1.5
mu_haptic = 6.0
sigma_haptic = 0.5

In [ ]:
# STUDENT CODE HERE
mu_multi = mu_haptic
sigma_multi = sigma_haptic

print_estimate("Combined multisensory estimate", mu_multi, sigma_multi)

In [ ]:
plot_gaussian_update(mu_visual, sigma_visual, mu_haptic, sigma_haptic, mu_multi, sigma_multi)
plt.title("Reliability-weighted multisensory integration")
plt.show()

**Questions**:
- Is the combined estimate exactly halfway between vision and haptics?
- Which sensory modality has higher precision?
- How does this connect to the lecture examples where one sense can bias another?

<a id="section5"></a>
## 5) Reliability and Bias

In the lecture, we saw that changing sensory reliability changes how strongly one
modality pulls perception. Here we will keep the visual and haptic estimates fixed,
but vary visual uncertainty.

**Student Task**: Complete the loop to compute the combined estimate for each
visual uncertainty. The loop currently uses only the haptic estimate.

In [11]:
visual_sigmas = np.linspace(0.25, 3.0, 30)
combined_means = []
combined_sigmas = []

In [ ]:
for sigma_v in visual_sigmas:
    # STUDENT CODE HERE
    mu_combined = mu_haptic
    sigma_combined = sigma_haptic
    combined_means.append(mu_combined)
    combined_sigmas.append(sigma_combined)

combined_means = np.array(combined_means)
combined_sigmas = np.array(combined_sigmas)

In [ ]:
plt.plot(visual_sigmas, combined_means, marker="o", label="Combined estimate")
plt.axhline(mu_visual, linestyle="--", color="tab:blue", label="Visual estimate")
plt.axhline(mu_haptic, linestyle="--", color="tab:orange", label="Haptic estimate")
plt.xlabel("Visual uncertainty, sigma_v (N)")
plt.ylabel("Combined weight estimate (N)")
plt.title("Less reliable vision gets less weight")
plt.legend()
plt.tight_layout()
plt.show()

**Questions**:
- What happens when visual uncertainty is very small?
- What happens when visual uncertainty is very large?
- In a rubber hand or mirror-box illusion, what might increase the weight given
  to vision?

<a id="section6"></a>
## 6) Discussion

Please edit this markdown cell to answer:

1. In your own words, what does it mean to weight a sensory estimate by
   reliability?
2. Why can a biased prior or a biased sensory cue produce a perceptual illusion?
3. How can repeated experience reduce the effect of an initially wrong prior?